# LeetCode 911 — Online Election bằng Python

Link đề: https://leetcode.com/problems/online-election/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Binary Search (tìm kiếm nhị phân), Design (thiết kế lớp)

Skeleton LeetCode:

```python
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        ...

    def q(self, t: int) -> int:
        ...


# Your TopVotedCandidate object will be instantiated and called as such:
# obj = TopVotedCandidate(persons, times)
# param_1 = obj.q(t)
```

Ghi chú: đây là bài **thiết kế lớp**. Hàm khởi tạo nhận toàn bộ lá phiếu một lần (đã biết trước), sau đó `q(t)` được gọi **nhiều lần**. Vì thế ta **tiền xử lý** một lần để biết **ai dẫn đầu sau mỗi lá phiếu**, rồi mỗi truy vấn chỉ cần **tìm nhị phân** xem lá phiếu cuối cùng có thời điểm `<= t` là lá nào. Quy tắc hòa phiếu ("lá phiếu gần nhất trong số những người hòa thắng") được xử lý rất gọn khi duyệt từng phiếu theo thứ tự: người vừa nhận phiếu mà **bằng hoặc hơn** số phiếu của người đang dẫn đầu sẽ trở thành người dẫn đầu.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` lá phiếu, lá thứ `i` bầu cho `persons[i]` tại thời điểm `times[i]` (tăng ngặt); truy vấn `q(t)` hỏi ai đang dẫn đầu tại thời điểm `t` (đếm cả các phiếu bỏ **đúng** tại `t`), hòa phiếu thì người có **lá phiếu gần nhất** thắng.
2. Nắm các công cụ Python cần thiết: `collections.Counter`, `bisect.bisect_right`, mảng song song (`times`, `wins`), và cách tự viết bộ chạy chuỗi lệnh kiểu LeetCode (`["TopVotedCandidate", "q", ...]`) để kiểm thử lớp.
3. Hiểu nguyên lý **tiền xử lý + tìm kiếm nhị phân**: kết quả của `q(t)` chỉ phụ thuộc vào **số lá phiếu đã bỏ tới thời điểm `t`**, tức là vị trí `bisect_right(times, t)`; dẫn đầu sau mỗi lá phiếu tính trước được trong một lượt `O(n)`.
4. Đi qua 4 cách: đếm lại từ đầu cho mỗi truy vấn (brute, đúng định nghĩa), tiền xử lý + `bisect` (cách nộp chính), tiền xử lý + tự viết tìm kiếm nhị phân, và **nén** chỉ giữ các thời điểm đổi người dẫn đầu; đối chiếu với một oracle độc lập.
5. Tránh các lỗi kinh điển: dùng `>` thay vì `>=` khi hòa phiếu, dùng `bisect_left` (bỏ sót phiếu đúng tại `t`), quên trừ `1` sau `bisect_right`, và tính lại phiếu cho từng truy vấn.

## 2. Đề bài & ví dụ

Trong một cuộc bầu cử, lá phiếu thứ `i` bầu cho `persons[i]` tại thời điểm `times[i]`.

Với mỗi truy vấn tại thời điểm `t`, tìm người **đang dẫn đầu** tại thời điểm `t`. Các phiếu bỏ **đúng tại `t`** được tính. Nếu **hòa**, người có **phiếu gần nhất** (trong số những người hòa) thắng.

Hãy cài đặt lớp `TopVotedCandidate`:

- `TopVotedCandidate(int[] persons, int[] times)`: khởi tạo đối tượng với hai mảng.
- `int q(int t)`: trả về người đang dẫn đầu tại thời điểm `t` theo luật trên.

**Ví dụ 1**

Input:

```
["TopVotedCandidate", "q", "q", "q", "q", "q", "q"]
[[[0, 1, 1, 0, 0, 1, 0], [0, 5, 10, 15, 20, 25, 30]], [3], [12], [25], [15], [24], [8]]
```

Output: `[null, 0, 1, 1, 0, 0, 1]`

Giải thích: `q(3)`: phiếu `[0]` nên `0` dẫn; `q(12)`: phiếu `[0,1,1]` nên `1` dẫn; `q(25)`: phiếu `[0,1,1,0,0,1]`, `0` và `1` hòa `3-3`, phiếu gần nhất (thời điểm `25`) là của `1` nên `1` thắng; `q(15)`: phiếu `[0,1,1,0]` hòa `2-2`, phiếu gần nhất là của `0` nên `0`; `q(24)`: phiếu `[0,1,1,0,0]`, `0` dẫn `3-2`; `q(8)`: phiếu `[0,1]` hòa `1-1`, phiếu gần nhất là của `1` nên `1`.

**Ràng buộc:**

- `1 <= persons.length <= 5000`
- `times.length == persons.length`
- `0 <= persons[i] < persons.length`
- `0 <= times[i] <= 10^9`
- `times` tăng **ngặt**.
- `times[0] <= t <= 10^9`
- Tối đa `10^4` lần gọi `q`.

In [1]:
from typing import List
from collections import Counter
from bisect import bisect_right, bisect_left
import random, time

def run_ops(cls, ops: List[str], args: list) -> list:
    """Mô phỏng cách LeetCode gọi lớp: phần tử đầu là tên lớp (constructor), các phần tử sau là tên method.
    Trả về list kết quả, None ứng với null."""
    obj = None
    out = []
    for i, (op, arg) in enumerate(zip(ops, args)):
        if i == 0:
            assert op == cls.__name__, (op, cls.__name__)
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

example_ops = ["TopVotedCandidate", "q", "q", "q", "q", "q", "q"]
example_args = [[[0, 1, 1, 0, 0, 1, 0], [0, 5, 10, 15, 20, 25, 30]], [3], [12], [25], [15], [24], [8]]
example_expected = [None, 0, 1, 1, 0, 0, 1]
print("ops     =", example_ops)
print("kỳ vọng =", example_expected)

ops     = ['TopVotedCandidate', 'q', 'q', 'q', 'q', 'q', 'q']
kỳ vọng = [None, 0, 1, 1, 0, 0, 1]


## 3. Kiến thức nền

### 3.1. Đếm phiếu và quy tắc hòa

Với `Counter` `cnt`, `cnt[p] += 1` thêm một phiếu cho `p`. Giả sử trước phiếu mới, `cur` đang dẫn đầu (có nhiều phiếu nhất; nếu hòa, người có phiếu gần nhất). Khi `p` nhận thêm phiếu:

- nếu `cnt[p] > cnt[cur]`: `p` vượt lên dẫn đầu;
- nếu `cnt[p] == cnt[cur]`: `p` **hòa** với `cur` và phiếu vừa bỏ cho `p` là **gần nhất** nên `p` thắng;
- nếu `cnt[p] < cnt[cur]`: `cur` vẫn dẫn.

Gộp lại: `if cnt[p] >= cnt[cur]: cur = p`.

In [2]:
cnt = Counter()
cur = None
for step, p in enumerate([0, 1, 1, 0, 0, 1, 0]):
    cnt[p] += 1
    if cur is None or cnt[p] >= cnt[cur]:
        cur = p
    print(f"phiếu {step} cho {p}: đếm {dict(cnt)} -> dẫn đầu {cur}")

phiếu 0 cho 0: đếm {0: 1} -> dẫn đầu 0
phiếu 1 cho 1: đếm {0: 1, 1: 1} -> dẫn đầu 1
phiếu 2 cho 1: đếm {0: 1, 1: 2} -> dẫn đầu 1
phiếu 3 cho 0: đếm {0: 2, 1: 2} -> dẫn đầu 0
phiếu 4 cho 0: đếm {0: 3, 1: 2} -> dẫn đầu 0
phiếu 5 cho 1: đếm {0: 3, 1: 3} -> dẫn đầu 1
phiếu 6 cho 0: đếm {0: 4, 1: 3} -> dẫn đầu 0


### 3.2. `bisect_right` và "phiếu cuối cùng có thời điểm `<= t`"

`bisect_right(times, t)` trả về số phần tử `<= t` trong `times` đã sắp xếp. Vì vậy `bisect_right(times, t) - 1` là **chỉ số của lá phiếu cuối cùng có `times[i] <= t`** (kể cả khi `times[i] == t`). Đây đúng là điều đề yêu cầu: các phiếu bỏ đúng tại `t` được tính. `bisect_left(times, t)` thì khác: nó trả về số phần tử `< t`, bỏ sót phiếu đúng tại `t`.

In [3]:
times = [0, 5, 10, 15, 20, 25, 30]
for t in (3, 5, 12, 25, 30, 100):
    print(f"t={t:3}: bisect_right-1 = {bisect_right(times, t) - 1} | bisect_left-1 = {bisect_left(times, t) - 1}")

t=  3: bisect_right-1 = 0 | bisect_left-1 = 0
t=  5: bisect_right-1 = 1 | bisect_left-1 = 0
t= 12: bisect_right-1 = 2 | bisect_left-1 = 2
t= 25: bisect_right-1 = 5 | bisect_left-1 = 4
t= 30: bisect_right-1 = 6 | bisect_left-1 = 5
t=100: bisect_right-1 = 6 | bisect_left-1 = 6


### 3.3. Bộ chạy chuỗi lệnh

Hàm `run_ops` ở mục 2 mô phỏng cách LeetCode kiểm tra bài thiết kế: phần tử đầu là tên lớp (gọi hàm khởi tạo), các phần tử sau là tên method với danh sách tham số tương ứng; kết quả `None` ứng với `null`.

### 3.4. Khai báo nhiều lần cùng tên lớp

Notebook sẽ định nghĩa `TopVotedCandidate` bốn lần (một cho mỗi cách). Sau mỗi lần ta **lưu tham chiếu** (ví dụ `cls_scan = TopVotedCandidate`) để dùng lại về sau; lần định nghĩa sau không làm mất đối tượng lớp đã lưu.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Với ví dụ 1: `persons = [0,1,1,0,0,1,0]`, `times = [0,5,10,15,20,25,30]`. Dẫn đầu **sau mỗi lá phiếu**:

| Phiếu | thời điểm | cho | đếm | dẫn đầu |
|---|---|---|---|---|
| 0 | 0 | 0 | `{0:1}` | 0 |
| 1 | 5 | 1 | `{0:1, 1:1}` hòa, phiếu gần nhất là của 1 | 1 |
| 2 | 10 | 1 | `{0:1, 1:2}` | 1 |
| 3 | 15 | 0 | `{0:2, 1:2}` hòa, gần nhất là 0 | 0 |
| 4 | 20 | 0 | `{0:3, 1:2}` | 0 |
| 5 | 25 | 1 | `{0:3, 1:3}` hòa, gần nhất là 1 | 1 |
| 6 | 30 | 0 | `{0:4, 1:3}` | 0 |

Truy vấn `q(12)`: các thời điểm `<= 12` là `0, 5, 10` (3 phiếu), nên lá phiếu cuối cùng là chỉ số `2`: dẫn đầu `1`. `q(25)`: thời điểm `<= 25` gồm cả `25`, lá phiếu cuối là chỉ số `5`: dẫn đầu `1`. `q(24)`: lá phiếu cuối chỉ số `4`: dẫn đầu `0`.

## 5. Cách 1 — Brute Force: đếm lại từ đầu cho mỗi truy vấn

**Ý tưởng.** Lưu `persons`, `times`. Mỗi lần `q(t)`: duyệt các phiếu `i` có `times[i] <= t`, đếm số phiếu của mỗi người và **thời điểm phiếu gần nhất**; chọn người có số phiếu nhiều nhất, hòa thì có phiếu gần nhất lớn nhất. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [4]:
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        self.persons = persons
        self.times = times

    def q(self, t: int) -> int:
        count = {}
        last = {}
        for p, tm in zip(self.persons, self.times):
            if tm > t:
                break
            count[p] = count.get(p, 0) + 1
            last[p] = tm
        # nhiều phiếu nhất; hòa thì phiếu gần nhất (thời điểm lớn nhất)
        return max(count, key=lambda p: (count[p], last[p]))

cls_scan = TopVotedCandidate
got = run_ops(cls_scan, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, 0, 1, 1, 0, 0, 1] OK


**Độ phức tạp và nút thắt.** `O(n)` mỗi truy vấn: với `5000` phiếu và `10^4` truy vấn là `5 * 10^7` thao tác. Xem thời gian:

In [5]:
rng = random.Random(911)
n = 2000
persons = [rng.randrange(n) % 50 for _ in range(n)]
times = sorted(rng.sample(range(10 ** 6), n))
obj = cls_scan(persons, times)
for qn in (100, 1000, 3000):
    qs = [rng.randint(times[0], 10 ** 6) for _ in range(qn)]
    t0 = time.perf_counter()
    for t in qs:
        obj.q(t)
    print(f"{qn:5} truy vấn, n={n}: {1000 * (time.perf_counter() - t0):8.1f} ms")

  100 truy vấn, n=2000:      9.6 ms
 1000 truy vấn, n=2000:     87.7 ms


 3000 truy vấn, n=2000:    278.0 ms


Thời gian tăng tuyến tính theo cả số truy vấn lẫn số phiếu. Nút thắt: ta **tính lại cùng một kết quả** cho mọi truy vấn có cùng tập phiếu (cùng khoảng giữa hai thời điểm bỏ phiếu liên tiếp).

## 6. Ý tưởng thuật toán / bước đột phá

**Kết quả `q(t)` chỉ phụ thuộc vào số phiếu đã bỏ tới thời điểm `t`.** Nếu `k = bisect_right(times, t)` là số phiếu có thời điểm `<= t` thì `q(t)` chính là **người dẫn đầu sau `k` phiếu đầu tiên**. Ta tính trước mảng `wins[i]` = người dẫn đầu sau lá phiếu thứ `i` trong **một lượt `O(n)`** nhờ quy tắc `if cnt[p] >= cnt[cur]: cur = p`; mỗi truy vấn chỉ cần một **tìm kiếm nhị phân** `O(log n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Tiền xử lý `wins` + `bisect_right` (nộp) |
| 3 | Tiền xử lý `wins` + **tự viết** tìm kiếm nhị phân |
| 4 | **Nén**: chỉ lưu thời điểm mà **người dẫn đầu đổi** |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tiền xử lý `wins` + `bisect_right` (cách nộp chính)

**Giải thích từng đoạn.**

- Trong `__init__`: `cnt = Counter()`, `cur = 0`, `wins = []`. Với mỗi phiếu `p`: `cnt[p] += 1`; nếu `cnt[cur] <= cnt[p]` thì `cur = p`; `wins.append(cur)`.
- Ban đầu `cur = 0` với `cnt[0] = 0`: phiếu đầu tiên cho `p` luôn thỏa `cnt[0] <= cnt[p] = 1`, nên `cur` trở thành `p` (đúng, vì chỉ có một người có phiếu). Trường hợp phiếu đầu cho chính `0` cũng đúng.
- `q(t)`: `i = bisect_right(self.times, t) - 1`, trả `self.wins[i]`.

In [6]:
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.times = times
        self.wins = []
        cur = 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        i = bisect_right(self.times, t) - 1
        return self.wins[i]

cls_bisect = TopVotedCandidate
got = run_ops(cls_bisect, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, 0, 1, 1, 0, 0, 1] OK


**Code nộp LeetCode (Cách 2):** chính là lớp ở trên (kèm `from bisect import bisect_right`, `from collections import Counter` ở đầu tệp).

```python
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.times = times
        self.wins = []
        cur = 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        i = bisect_right(self.times, t) - 1
        return self.wins[i]
```

**Độ phức tạp (Cách 2):** khởi tạo `O(n)`; mỗi `q` là `O(log n)`; bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Tiền xử lý `wins` + tự viết tìm kiếm nhị phân

**Ý tưởng khác.** Không dùng `bisect`: tự tìm **chỉ số lớn nhất `i` với `times[i] <= t`** bằng nhị phân thủ công. Dùng cận `lo = 0`, `hi = n - 1`, giữ bất biến "`times[lo] <= t`" (đúng vì `t >= times[0]`): lặp `while lo < hi`: `mid = (lo + hi + 1) // 2` (lấy trung bình **làm tròn lên** để không vòng lặp vô hạn); nếu `times[mid] <= t` thì `lo = mid` ngược lại `hi = mid - 1`.

**Giải thích từng đoạn.** Làm tròn lên ở `mid` là bắt buộc khi gán `lo = mid`; nếu làm tròn xuống thì với `hi = lo + 1` ta có `mid = lo` và `lo = mid` không tiến triển.

In [7]:
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        self.times = times
        self.wins = []
        count = {}
        cur = 0
        for p in persons:
            count[p] = count.get(p, 0) + 1
            if count.get(cur, 0) <= count[p]:
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        lo, hi = 0, len(self.times) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if self.times[mid] <= t:
                lo = mid
            else:
                hi = mid - 1
        return self.wins[lo]

cls_manual = TopVotedCandidate
got = run_ops(cls_manual, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, 0, 1, 1, 0, 0, 1] OK


**Code nộp LeetCode (Cách 3):** chính là lớp ở trên.

**Độ phức tạp (Cách 3):** khởi tạo `O(n)`; mỗi `q` là `O(log n)`; bộ nhớ `O(n)`. Điều kiện `t >= times[0]` của đề bảo đảm `lo = 0` là hợp lệ.

---

### 7.3. Cách 4 — Nén: chỉ lưu các thời điểm đổi người dẫn đầu

**Ý tưởng khác.** Nhiều phiếu liên tiếp không làm đổi người dẫn đầu. Ta chỉ lưu `change_times[j]` là thời điểm phiếu làm **người dẫn đầu đổi** (hoặc phiếu đầu tiên) và `change_leader[j]` là người dẫn đầu từ lúc đó. Truy vấn `q(t)`: `bisect_right(change_times, t) - 1`.

**Giải thích từng đoạn.** Khi `p` vượt/hòa người dẫn đầu và `p != cur` thì ghi nhận một thay đổi; nếu `p == cur` thì người dẫn đầu giữ nguyên nên không thêm. Kết quả tại mọi `t` bằng Cách 2 nhưng mảng có thể ngắn hơn nhiều (ví dụ một người luôn dẫn đầu thì chỉ có một mục).

In [8]:
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.change_times = []
        self.change_leader = []
        cur = None
        for p, tm in zip(persons, times):
            cnt[p] += 1
            if cur is None or cnt[p] >= cnt[cur]:
                if p != cur:
                    self.change_times.append(tm)
                    self.change_leader.append(p)
                cur = p

    def q(self, t: int) -> int:
        i = bisect_right(self.change_times, t) - 1
        return self.change_leader[i]

cls_compressed = TopVotedCandidate
got = run_ops(cls_compressed, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected
print("số mục đã nén ở ví dụ 1:", len(cls_compressed(*example_args[0]).change_times), "/ 7 phiếu")

[None, 0, 1, 1, 0, 0, 1] OK
số mục đã nén ở ví dụ 1: 5 / 7 phiếu


**Code nộp LeetCode (Cách 4):** chính là lớp ở trên.

**Độ phức tạp (Cách 4):** khởi tạo `O(n)`; mỗi `q` là `O(log c)` với `c <= n` là số lần đổi người dẫn đầu; bộ nhớ `O(c)`.

## 8. TRACE từng bước

### 8.1. Cách 2: khởi tạo và truy vấn cho ví dụ 1

In [9]:
def trace_init(persons, times):
    cnt = Counter()
    cur = 0
    wins = []
    for p, tm in zip(persons, times):
        before = cur
        cnt[p] += 1
        if cnt[cur] <= cnt[p]:
            cur = p
        wins.append(cur)
        print(f"thời điểm {tm:2}: phiếu cho {p}; đếm {dict(cnt)}; dẫn đầu {before} -> {cur}")
    return wins

wins = trace_init(*example_args[0])
print("wins =", wins)
assert wins == [0, 1, 1, 0, 0, 1, 0]

thời điểm  0: phiếu cho 0; đếm {0: 1}; dẫn đầu 0 -> 0
thời điểm  5: phiếu cho 1; đếm {0: 1, 1: 1}; dẫn đầu 0 -> 1
thời điểm 10: phiếu cho 1; đếm {0: 1, 1: 2}; dẫn đầu 1 -> 1
thời điểm 15: phiếu cho 0; đếm {0: 2, 1: 2}; dẫn đầu 1 -> 0
thời điểm 20: phiếu cho 0; đếm {0: 3, 1: 2}; dẫn đầu 0 -> 0
thời điểm 25: phiếu cho 1; đếm {0: 3, 1: 3}; dẫn đầu 0 -> 1
thời điểm 30: phiếu cho 0; đếm {0: 4, 1: 3}; dẫn đầu 1 -> 0
wins = [0, 1, 1, 0, 0, 1, 0]


In [10]:
times = example_args[0][1]
for t in (3, 12, 25, 15, 24, 8):
    k = bisect_right(times, t)
    print(f"q({t:2}): bisect_right = {k} -> lá phiếu cuối chỉ số {k - 1} -> dẫn đầu {wins[k - 1]}")
assert [wins[bisect_right(times, t) - 1] for t in (3, 12, 25, 15, 24, 8)] == example_expected[1:]

q( 3): bisect_right = 1 -> lá phiếu cuối chỉ số 0 -> dẫn đầu 0
q(12): bisect_right = 3 -> lá phiếu cuối chỉ số 2 -> dẫn đầu 1
q(25): bisect_right = 6 -> lá phiếu cuối chỉ số 5 -> dẫn đầu 1
q(15): bisect_right = 4 -> lá phiếu cuối chỉ số 3 -> dẫn đầu 0
q(24): bisect_right = 5 -> lá phiếu cuối chỉ số 4 -> dẫn đầu 0
q( 8): bisect_right = 2 -> lá phiếu cuối chỉ số 1 -> dẫn đầu 1


### 8.2. Cách 3 (nhị phân thủ công) cho `q(24)`

`times = [0,5,10,15,20,25,30]`, `t = 24`:

- `lo = 0`, `hi = 6`: `mid = 3`, `times[3] = 15 <= 24` ⇒ `lo = 3`.
- `lo = 3`, `hi = 6`: `mid = 5`, `times[5] = 25 > 24` ⇒ `hi = 4`.
- `lo = 3`, `hi = 4`: `mid = 4`, `times[4] = 20 <= 24` ⇒ `lo = 4`.
- `lo = hi = 4`: dừng. `wins[4] = 0`.

In [11]:
def trace_manual_search(times, t):
    lo, hi = 0, len(times) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok = times[mid] <= t
        print(f"lo={lo} hi={hi} mid={mid} times[mid]={times[mid]} {'<=' if ok else '>'} {t}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    print("kết quả chỉ số:", lo)
    return lo

assert trace_manual_search(times, 24) == 4

lo=0 hi=6 mid=3 times[mid]=15 <= 24
lo=3 hi=6 mid=5 times[mid]=25 > 24
lo=3 hi=4 mid=4 times[mid]=20 <= 24
kết quả chỉ số: 4


### 8.3. Cách 4 (nén) cho ví dụ 1

Người dẫn đầu sau mỗi phiếu là `[0, 1, 1, 0, 0, 1, 0]`: đổi tại các thời điểm `0, 5, 15, 25, 30`. Vậy `change_times = [0, 5, 15, 25, 30]`, `change_leader = [0, 1, 0, 1, 0]`; phiếu tại `10` và `20` không đổi người dẫn đầu nên không lưu.

In [12]:
obj = cls_compressed(*example_args[0])
print(obj.change_times, obj.change_leader)
assert (obj.change_times, obj.change_leader) == ([0, 5, 15, 25, 30], [0, 1, 0, 1, 0])

[0, 5, 15, 25, 30] [0, 1, 0, 1, 0]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (tiền xử lý `wins` + `bisect_right`): ngắn gọn, `O(n)` khởi tạo, `O(log n)` mỗi truy vấn.

```python
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.times = times
        self.wins = []
        cur = 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        i = bisect_right(self.times, t) - 1
        return self.wins[i]
```

In [13]:
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.times = times
        self.wins = []
        cur = 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        i = bisect_right(self.times, t) - 1
        return self.wins[i]

cls_final = TopVotedCandidate
got = run_ops(cls_final, example_ops, example_args)
print(got, "OK" if got == example_expected else "SAI")
assert got == example_expected

[None, 0, 1, 1, 0, 0, 1] OK


## 10. Vì sao đúng

**(a) `q(t)` chỉ phụ thuộc vào số phiếu đã bỏ.** Các lá phiếu bỏ tại thời điểm `<= t` là một **tiền tố** của dãy phiếu (vì `times` tăng ngặt), có `k = bisect_right(times, t)` phiếu. Đề định nghĩa người dẫn đầu là người có nhiều phiếu nhất trong `k` phiếu đó, hòa thì người có phiếu gần nhất; kết quả chỉ phụ thuộc `k`.

**(b) Cập nhật `if cnt[cur] <= cnt[p]: cur = p` đúng.** Quy nạp theo số phiếu. Cơ sở: sau `0` phiếu không có ai (dùng `cur = 0` với `cnt[0] = 0` chỉ để thống nhất; phiếu đầu tiên luôn làm `cur = p`). Bước quy nạp: giả sử `cur` là người dẫn đầu đúng sau `j` phiếu (nhiều phiếu nhất; nếu hòa thì gần nhất). Phiếu thứ `j + 1` cho `p`: số phiếu của mọi người **khác `p` không đổi**, của `p` tăng `1`.
   - Nếu `cnt[p] > cnt[cur]` (sau khi tăng): `p` có nhiều phiếu nhất thực sự (nhiều hơn `cur`, mà `cur` đã nhiều hơn hoặc bằng mọi người khác): `p` dẫn.
   - Nếu `cnt[p] == cnt[cur]`: `p` hòa với `cur` về số phiếu cao nhất; phiếu vừa rồi (của `p`) là **gần nhất** trong cả cuộc bầu cử, nên `p` thắng.
   - Nếu `cnt[p] < cnt[cur]`: `cur` vẫn có nhiều phiếu nhất; những người hòa với `cur` không thay đổi và `cur` vẫn là người có phiếu gần nhất trong số họ vì phiếu mới thuộc `p`, không thuộc nhóm hòa. Vậy `cur` vẫn dẫn.
   Trường hợp `p == cur` thuộc nhánh `>`.

**(c) `bisect_right - 1`.** `bisect_right(times, t)` là số phần tử `<= t`; lá phiếu cuối cùng có `times[i] <= t` ở chỉ số `bisect_right(times, t) - 1`; đề đảm bảo `t >= times[0]` nên chỉ số `>= 0`.

**(d) Cách 3.** Nhị phân thủ công tìm chỉ số lớn nhất `i` có `times[i] <= t` với bất biến `times[lo] <= t` và mọi chỉ số `> hi` có `times > t`; `mid` làm tròn lên bảo đảm hội tụ.

**(e) Cách 4 (nén).** Chỉ khi `cur` đổi mới ghi lại; mảng `wins` đầy đủ là hàm hằng từng khúc nên giá trị tại thời điểm `t` bằng giá trị của lần đổi gần nhất có thời điểm `<= t`.

**Kiểm chứng bằng máy.** Oracle **độc lập**: tính từ đầu cho mỗi truy vấn bằng cách đếm tất cả phiếu có `times <= t` trong một `dict` và chọn `max` theo khóa `(số phiếu, thời điểm phiếu gần nhất)`, dùng một mô hình khác hẳn quy tắc `>=` (Cách 1 chính là oracle này). So với mọi lớp bằng bộ chạy chuỗi lệnh.

In [14]:
def oracle_answer(persons, times, queries):
    res = []
    for t in queries:
        cnt, last = {}, {}
        for p, tm in zip(persons, times):
            if tm <= t:
                cnt[p] = cnt.get(p, 0) + 1
                last[p] = tm
        res.append(max(cnt, key=lambda p: (cnt[p], last[p])))
    return res

# kiểm tra oracle trên ví dụ đề (kỳ vọng đã cho trong đề)
assert oracle_answer(*example_args[0], [a[0] for a in example_args[1:]]) == example_expected[1:]
print("oracle khớp ví dụ đề")

oracle khớp ví dụ đề


## 11. Time / Space Complexity

| Cách | Khởi tạo | Mỗi `q` | Space |
|---|---|---|---|
| 1. Đếm lại | `O(1)` | `O(n)` | `O(n)` |
| 2. `wins` + `bisect` | `O(n)` | `O(log n)` | `O(n)` |
| 3. `wins` + nhị phân tay | `O(n)` | `O(log n)` | `O(n)` |
| 4. Nén thay đổi | `O(n)` | `O(log c)` | `O(c)` |

Đo thử với `n = 5000` phiếu và `10^4` truy vấn (cỡ tối đa):

In [15]:
rng = random.Random(11)
n = 5000
persons = [rng.randrange(n) for _ in range(n)]
times = sorted(rng.sample(range(10 ** 9 + 1), n))
qs = [rng.randint(times[0], 10 ** 9) for _ in range(10 ** 4)]
ref = oracle_answer(persons, times, qs[:300])
for name, cls in (("Cách 2 bisect", cls_bisect), ("Cách 3 nhị phân tay", cls_manual), ("Cách 4 nén", cls_compressed), ("nộp", cls_final)):
    t0 = time.perf_counter()
    obj = cls(persons, times)
    res = [obj.q(t) for t in qs]
    dt = (time.perf_counter() - t0) * 1000
    assert res[:300] == ref, name
    print(f"{name:20} khởi tạo + 10^4 truy vấn: {dt:8.1f} ms")
t0 = time.perf_counter()
obj = cls_scan(persons, times)
[obj.q(t) for t in qs[:300]]
print(f"{'Cách 1 đếm lại':20} chỉ 300 truy vấn: {1000 * (time.perf_counter() - t0):8.1f} ms (10^4 truy vấn sẽ gấp ~33 lần)")

Cách 2 bisect        khởi tạo + 10^4 truy vấn:      4.1 ms
Cách 3 nhị phân tay  khởi tạo + 10^4 truy vấn:     13.8 ms
Cách 4 nén           khởi tạo + 10^4 truy vấn:      2.6 ms
nộp                  khởi tạo + 10^4 truy vấn:      4.1 ms
Cách 1 đếm lại       chỉ 300 truy vấn:    145.9 ms (10^4 truy vấn sẽ gấp ~33 lần)


## 12. Bộ test

Gồm: ví dụ của đề (qua bộ chạy chuỗi lệnh), edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi lớp với oracle, test ngẫu nhiên có seed (đúng ràng buộc: `times` tăng ngặt, `persons[i] < n`, `t >= times[0]`) và tính chất.

In [16]:
classes = {
    "scan": cls_scan,
    "bisect": cls_bisect,
    "manual": cls_manual,
    "compressed": cls_compressed,
    "final": cls_final,
}

def run_all(persons, times, queries, expected=None):
    ref = oracle_answer(persons, times, queries)
    ops = ["TopVotedCandidate"] + ["q"] * len(queries)
    args = [[list(persons), list(times)]] + [[t] for t in queries]
    for name, cls in classes.items():
        out = run_ops(cls, ops, args)
        assert out == [None] + ref, (name, persons, times, queries, out)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", persons, times, queries, expected, ref)
    return ref

# 12.1 ví dụ đề qua bộ chạy chuỗi lệnh
for name, cls in classes.items():
    assert run_ops(cls, example_ops, example_args) == example_expected, name
print("12.1 ví dụ đề qua bộ chạy chuỗi lệnh: OK cho", list(classes))

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0], [1], [1, 2, 10 ** 9], [0, 0, 0]),                          # một phiếu; truy vấn đúng tại thời điểm phiếu và sau đó
    ([0, 1], [1, 2], [1, 2, 3], [0, 1, 1]),                          # hòa 1-1: phiếu gần nhất thắng
    ([0, 0, 1], [1, 2, 3], [1, 2, 3, 100], [0, 0, 0, 0]),            # 0 dẫn 2-1 rồi 2-1 (1 chỉ có một phiếu)
    ([1, 0, 0, 1], [2, 4, 6, 8], [2, 3, 4, 5, 6, 7, 8, 9], [1, 1, 0, 0, 0, 0, 1, 1]),   # hòa 1-1 lúc 4 (gần nhất 0); 0 dẫn 2-1 lúc 6; hòa 2-2 lúc 8 (gần nhất 1)
    ([2, 2, 1, 1], [1, 3, 4, 6], [1, 2, 3, 4, 5, 6, 7], [2, 2, 2, 2, 2, 1, 1]),
    ([0, 1, 0, 1], [10, 20, 30, 40], [10, 15, 20, 29, 30, 39, 40], [0, 0, 1, 1, 0, 0, 1]),
]
for persons, times, queries, expected in edge:
    run_all(persons, times, queries, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề qua bộ chạy chuỗi lệnh: OK cho ['scan', 'bisect', 'manual', 'compressed', 'final']
12.2 edge case: OK 6 ca


In [17]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
class WrongStrict:                                   # dùng > thay vì >= (hòa phiếu giữ người dẫn cũ)
    def __init__(self, persons, times):
        cnt = Counter()
        self.times, self.wins, cur = times, [], 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] < cnt[p]:
                cur = p
            self.wins.append(cur)
    def q(self, t):
        return self.wins[bisect_right(self.times, t) - 1]

class WrongBisectLeft:                               # bisect_left: bỏ sót phiếu bỏ đúng tại t
    def __init__(self, persons, times):
        cnt = Counter()
        self.times, self.wins, cur = times, [], 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)
    def q(self, t):
        return self.wins[max(bisect_left(self.times, t) - 1, 0)]

class WrongNoMinusOne:                               # quên trừ 1 sau bisect_right
    def __init__(self, persons, times):
        cnt = Counter()
        self.times, self.wins, cur = times, [], 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)
    def q(self, t):
        return self.wins[bisect_right(self.times, t)]

class WrongMaxCounter:                               # dùng max trên Counter (hòa thì lấy người xuất hiện sớm nhất)
    def __init__(self, persons, times):
        cnt = Counter()
        self.times, self.wins = times, []
        for p in persons:
            cnt[p] += 1
            self.wins.append(max(cnt, key=cnt.get))
    def q(self, t):
        return self.wins[bisect_right(self.times, t) - 1]

def head(r):
    return r if isinstance(r, str) else r[:8]

def attempt(cls, persons, times, queries):
    try:
        obj = cls(list(persons), list(times))
        return [obj.q(t) for t in queries]
    except Exception as e:
        return f"lỗi {type(e).__name__}"

def find_counter(cls, tries=20000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 7)
        persons = [rng.randrange(n) % 3 for _ in range(n)]
        times = sorted(rng.sample(range(1, 30), n))
        queries = [rng.randint(times[0], 30) for _ in range(6)] + times
        if attempt(cls, persons, times, queries) != oracle_answer(persons, times, queries):
            return persons, times, queries
    return None

counters = {}
for name, cls in (("dùng > khi hòa", WrongStrict), ("bisect_left", WrongBisectLeft),
                  ("quên trừ 1", WrongNoMinusOne), ("max trên Counter", WrongMaxCounter)):
    c = find_counter(cls)
    assert c is not None, name
    counters[name] = c
    persons, times, queries = c
    print(f"{name:20} persons={persons} times={times}: sai = {head(attempt(cls, persons, times, queries))}, đúng = {oracle_answer(persons, times, queries)[:8]}")
    run_all(persons, times, queries)
print("12.3 phản ví dụ: OK")

dùng > khi hòa       persons=[1, 2, 2, 1, 1] times=[1, 11, 17, 22, 28]: sai = [1, 1, 2, 1, 1, 2, 1, 1], đúng = [2, 1, 2, 1, 2, 2, 1, 2]
bisect_left          persons=[1, 2, 2, 1, 1] times=[1, 11, 17, 22, 28]: sai = [2, 1, 2, 1, 1, 2, 1, 1], đúng = [2, 1, 2, 1, 2, 2, 1, 2]
quên trừ 1           persons=[1, 2, 2, 1, 1] times=[1, 11, 17, 22, 28]: sai = lỗi IndexError, đúng = [2, 1, 2, 1, 2, 2, 1, 2]
max trên Counter     persons=[1, 2, 2, 1, 1] times=[1, 11, 17, 22, 28]: sai = [1, 1, 2, 1, 1, 2, 1, 1], đúng = [2, 1, 2, 1, 2, 2, 1, 2]
12.3 phản ví dụ: OK


In [18]:
# 12.4 ngẫu nhiên nhỏ: so mọi lớp với oracle (n <= 12; truy vấn tại mọi thời điểm phiếu, ngay trước/sau và ngẫu nhiên)
rng = random.Random(4911)
for _ in range(1500):
    n = rng.randint(1, 12)
    persons = [rng.randrange(n) % rng.randint(1, 4) for _ in range(n)]
    times = sorted(rng.sample(range(0, 60), n))
    queries = times + [x - 1 for x in times if x - 1 >= times[0]] + [x + 1 for x in times] + [rng.randint(times[0], 100) for _ in range(5)]
    run_all(persons, times, queries)
print("12.4 ngẫu nhiên n <= 12 (1500 ca): OK")

# 12.5 đúng ràng buộc: n tới 5000, persons[i] < n, times tăng ngặt trong [0, 10^9], tới 10^4 truy vấn; so các lớp nhanh với nhau và oracle trên mẫu
rng = random.Random(5911)
for n, ncand in ((5000, 5000), (5000, 3), (2000, 50), (300, 2), (1, 1)):
    persons = [rng.randrange(n) % ncand for _ in range(n)]
    times = sorted(rng.sample(range(10 ** 9 + 1), n))
    queries = [rng.randint(times[0], 10 ** 9) for _ in range(10 ** 4)] + times[: 200] + [10 ** 9]
    outs = {}
    for name in ("bisect", "manual", "compressed", "final"):
        obj = classes[name](list(persons), list(times))
        outs[name] = [obj.q(t) for t in queries]
    assert len({tuple(v) for v in outs.values()}) == 1, (n, ncand)
    sample = rng.sample(range(len(queries)), 150)
    assert [outs["final"][i] for i in sample] == oracle_answer(persons, times, [queries[i] for i in sample])
print("12.5 n tới 5000, 10^4 truy vấn: OK")

# 12.6 tính chất: kết quả tại thời điểm t luôn là người đã được bầu ít nhất một phiếu; sau phiếu cuối không đổi
rng = random.Random(6911)
for _ in range(300):
    n = rng.randint(1, 20)
    persons = [rng.randrange(n) % rng.randint(1, 5) for _ in range(n)]
    times = sorted(rng.sample(range(0, 200), n))
    obj = cls_final(persons, times)
    for t in range(times[0], 210):
        r = obj.q(t)
        voted = {p for p, tm in zip(persons, times) if tm <= t}
        assert r in voted
        if t >= times[-1]:
            assert r == obj.q(times[-1])
print("12.6 tính chất: OK")

# 12.7 mọi truy vấn trong một khoảng giữa hai phiếu liên tiếp có cùng kết quả
rng = random.Random(7911)
for _ in range(200):
    n = rng.randint(2, 15)
    persons = [rng.randrange(n) % 3 for _ in range(n)]
    times = sorted(rng.sample(range(0, 100), n))
    obj = cls_final(persons, times)
    for a, b in zip(times, times[1:]):
        assert len({obj.q(t) for t in range(a, b)}) == 1
print("12.7 hằng số từng khúc giữa các phiếu: OK")

12.4 ngẫu nhiên n <= 12 (1500 ca): OK


12.5 n tới 5000, 10^4 truy vấn: OK
12.6 tính chất: OK
12.7 hằng số từng khúc giữa các phiếu: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>` thay vì `>=` khi hòa phiếu

Khi hòa, **người vừa nhận phiếu** (phiếu gần nhất) phải thắng. Dùng `>` giữ người dẫn cũ.

In [19]:
persons, times, queries = counters["dùng > khi hòa"]
print("persons =", persons, "times =", times)
print("sai    ->", head(attempt(WrongStrict, persons, times, queries)))
print("đúng   ->", head(attempt(cls_final, persons, times, queries)))
assert attempt(WrongStrict, persons, times, queries) != attempt(cls_final, persons, times, queries)
wrong_obj = WrongStrict(*example_args[0])
print("ví dụ đề với >:", [wrong_obj.q(a[0]) for a in example_args[1:]], "(sai) so với", example_expected[1:])

persons = [1, 2, 2, 1, 1] times = [1, 11, 17, 22, 28]
sai    -> [1, 1, 2, 1, 1, 2, 1, 1]
đúng   -> [2, 1, 2, 1, 2, 2, 1, 2]
ví dụ đề với >: [0, 1, 0, 1, 0, 0] (sai) so với [0, 1, 1, 0, 0, 1]


Vì sao sai: ví dụ đề `q(25)` có `0` và `1` hòa `3-3` và `1` có phiếu gần nhất. **Đúng:** `if cnt[cur] <= cnt[p]: cur = p`.

### Lỗi 2 — `bisect_left` (bỏ sót phiếu đúng tại `t`)

Đề nói phiếu bỏ **tại thời điểm `t`** được tính. `bisect_left(times, t) - 1` bỏ qua phiếu `times[i] == t`.

In [20]:
persons, times, queries = counters["bisect_left"]
print("persons =", persons, "times =", times)
print("sai    ->", head(attempt(WrongBisectLeft, persons, times, queries)))
print("đúng   ->", head(attempt(cls_final, persons, times, queries)))
assert attempt(WrongBisectLeft, persons, times, queries) != attempt(cls_final, persons, times, queries)

persons = [1, 2, 2, 1, 1] times = [1, 11, 17, 22, 28]
sai    -> [2, 1, 2, 1, 1, 2, 1, 1]
đúng   -> [2, 1, 2, 1, 2, 2, 1, 2]


Vì sao sai: tại đúng thời điểm một phiếu, người nhận phiếu có thể đổi người dẫn đầu. **Đúng:** `bisect_right(times, t) - 1`.

### Lỗi 3 — Quên trừ `1` sau `bisect_right`

`bisect_right` trả về **số lượng** phần tử `<= t`, không phải chỉ số của phần tử cuối.

In [21]:
persons, times, queries = counters["quên trừ 1"]
print("persons =", persons, "times =", times)
print("sai    ->", head(attempt(WrongNoMinusOne, persons, times, queries)))
print("đúng   ->", head(attempt(cls_final, persons, times, queries)))
assert attempt(WrongNoMinusOne, persons, times, queries) != attempt(cls_final, persons, times, queries)

persons = [1, 2, 2, 1, 1] times = [1, 11, 17, 22, 28]
sai    -> lỗi IndexError
đúng   -> [2, 1, 2, 1, 2, 2, 1, 2]


Vì sao sai: lấy `wins[k]` thay vì `wins[k - 1]` cho kết quả lệch một phiếu, và `IndexError` khi `t` sau phiếu cuối. **Đúng:** `wins[bisect_right(times, t) - 1]`.

### Lỗi 4 — Dùng `max` trên `Counter` để tìm người dẫn đầu

`max(cnt, key=cnt.get)` khi hòa trả về người **xuất hiện sớm nhất** trong từ điển, không phải người có **phiếu gần nhất**.

In [22]:
persons, times, queries = counters["max trên Counter"]
print("persons =", persons, "times =", times)
print("sai    ->", head(attempt(WrongMaxCounter, persons, times, queries)))
print("đúng   ->", head(attempt(cls_final, persons, times, queries)))
assert attempt(WrongMaxCounter, persons, times, queries) != attempt(cls_final, persons, times, queries)

persons = [1, 2, 2, 1, 1] times = [1, 11, 17, 22, 28]
sai    -> [1, 1, 2, 1, 1, 2, 1, 1]
đúng   -> [2, 1, 2, 1, 2, 2, 1, 2]


Vì sao sai: quy tắc hòa phụ thuộc **thứ tự bỏ phiếu**, không phải thứ tự trong từ điển. **Đúng:** theo dõi người dẫn đầu và dùng `>=` khi cập nhật theo từng phiếu.

### Lỗi 5 — Tính lại số phiếu cho mỗi truy vấn

Không sai kết quả nhưng `O(n)` mỗi truy vấn: với `5000` phiếu và `10^4` truy vấn là `5 * 10^7` phép, dễ vượt thời gian (xem mục 5 và 11). **Đúng:** tính trước `wins` trong `__init__`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** bài **thiết kế lớp** với dữ liệu cho trước và nhiều truy vấn "tại thời điểm `t`" ⇒ **tiền xử lý trong `__init__`** và **tìm kiếm nhị phân** trong truy vấn.
- **Người dẫn đầu sau mỗi phiếu:** `if cnt[p] >= cnt[cur]: cur = p` (hòa thì người mới nhất thắng).
- `bisect_right(times, t) - 1` = lá phiếu cuối có thời điểm `<= t` (tính cả đúng `t`).
- Kiểm thử bài thiết kế bằng bộ chạy chuỗi lệnh `run_ops`.
- Cùng họ: Time Based Key-Value Store (981), Snapshot Array (1146), My Calendar I (729), Design Hit Counter (362).
- Test nên có: một phiếu, truy vấn đúng tại thời điểm phiếu và ngay trước/sau, hòa phiếu nhiều lần, một người dẫn đầu suốt cuộc bầu cử.

## 15. Biến thể

1. **Hòa phiếu chọn người có mã nhỏ nhất** (thay vì phiếu gần nhất): đổi điều kiện cập nhật.
2. **Tổng thời gian mỗi người dẫn đầu** trong một khoảng (bài tập 5).
3. **Truy vấn top-k** hoặc **số phiếu hiện tại** của người dẫn đầu: lưu thêm `cnt`.
4. Bài liên quan: Time Based Key-Value Store (981).

Dưới đây là biến thể 1 (hòa chọn mã nhỏ nhất) kiểm bằng oracle riêng.

In [23]:
class TopVotedSmallestIdTie:
    def __init__(self, persons, times):
        cnt = Counter()
        self.times, self.wins, cur = times, [], None
        for p in persons:
            cnt[p] += 1
            if cur is None or cnt[p] > cnt[cur] or (cnt[p] == cnt[cur] and p < cur):
                cur = p
            self.wins.append(cur)
    def q(self, t):
        return self.wins[bisect_right(self.times, t) - 1]

def oracle_smallest_id(persons, times, t):
    cnt = Counter(p for p, tm in zip(persons, times) if tm <= t)
    best = max(cnt.values())
    return min(p for p, c in cnt.items() if c == best)

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 10)
    persons = [rng.randrange(n) % 4 for _ in range(n)]
    times = sorted(rng.sample(range(0, 50), n))
    obj = TopVotedSmallestIdTie(persons, times)
    for t in range(times[0], 55):
        assert obj.q(t) == oracle_smallest_id(persons, times, t), (persons, times, t)
print("biến thể 1 (hòa chọn mã nhỏ nhất) khớp oracle")

biến thể 1 (hòa chọn mã nhỏ nhất) khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Khởi tạo | Mỗi `q` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Đếm lại | Theo định nghĩa | `O(1)` | `O(n)` | `O(n)` | Oracle |
| 2. `wins` + `bisect` | Tiền xử lý + nhị phân | `O(n)` | `O(log n)` | `O(n)` | **Cách nộp chính** |
| 3. Nhị phân tay | Không thư viện | `O(n)` | `O(log n)` | `O(n)` | Khi không được dùng `bisect` |
| 4. Nén | Chỉ lưu lần đổi người dẫn | `O(n)` | `O(log c)` | `O(c)` | `c` nhỏ hơn nhiều `n` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(bisect_right([0, 5, 10], 5), bisect_left([0, 5, 10], 5))
print(bisect_right([0, 5, 10], 7) - 1)
cnt = Counter([0, 1, 1]); print(cnt[0], cnt[1], cnt[2])
```

In [24]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `persons = [2, 2, 1, 1]`, `times = [1, 3, 4, 6]`: ghi kết quả `q(t)` cho `t = 1, 2, 3, 4, 5, 6, 7`.

In [25]:
# Bài 2: điền kết quả tính tay
hand_answers = None           # danh sách 7 phần tử

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành lớp với `wins` và `bisect_right`.

In [26]:
class ExerciseTopVoted3:

    def __init__(self, persons: List[int], times: List[int]):
        self.times = times
        self.wins = []
        cnt = Counter()
        cur = 0
        for p in persons:
            cnt[p] += 1
            # TODO 1: cập nhật cur khi cnt[p] >= cnt[cur]
            self.wins.append(cur)

    def q(self, t: int) -> int:
        # TODO 2: tìm chỉ số lá phiếu cuối cùng có thời điểm <= t và trả về người dẫn đầu
        pass


# obj = ExerciseTopVoted3(*example_args[0])
# assert [obj.q(a[0]) for a in example_args[1:]] == example_expected[1:]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết lớp `ExerciseTopVoted4` chỉ lưu các thời điểm đổi người dẫn đầu.

In [27]:
class ExerciseTopVoted4:

    def __init__(self, persons: List[int], times: List[int]):
        # TODO
        pass

    def q(self, t: int) -> int:
        # TODO
        pass


# obj = ExerciseTopVoted4(*example_args[0])
# assert [obj.q(a[0]) for a in example_args[1:]] == example_expected[1:]

### Bài 5 — Mini challenge: tổng thời gian dẫn đầu (khó)

Viết `exercise_5_durations(persons, times, end)` trả về `dict` ánh xạ `người -> tổng số đơn vị thời gian nguyên` mà người đó dẫn đầu trong các thời điểm `t = times[0], times[0] + 1, ..., end - 1` (mỗi `t` được tính một đơn vị, dẫn đầu theo `q(t)`). Chỉ gồm các người có thời gian `> 0`.

```python
exercise_5_durations([2, 2, 1, 1], [1, 3, 4, 6], 8)                      -> {2: 5, 1: 2}
exercise_5_durations([0, 1, 1, 0, 0, 1, 0], [0, 5, 10, 15, 20, 25, 30], 35) -> {0: 20, 1: 15}
```
Gợi ý: dùng `wins` và các khoảng `[times[i], times[i + 1])`.

In [28]:
def exercise_5_durations(persons: List[int], times: List[int], end: int) -> dict:
    # TODO
    pass


# assert exercise_5_durations([2, 2, 1, 1], [1, 3, 4, 6], 8) == {2: 5, 1: 2}
# assert exercise_5_durations([0, 1, 1, 0, 0, 1, 0], [0, 5, 10, 15, 20, 25, 30], 35) == {0: 20, 1: 15}

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `bisect_right` đếm số phần tử `<= x`; `bisect_left` đếm số phần tử `< x`.
- Mức 2: `[0, 5, 10]` và `5`: bên phải `2`, bên trái `1`; với `7`: `bisect_right = 2`, trừ `1`.
- Mức 3: `2 1`, `1`, `1 2 0`.

**Bài 2**
- Mức 1: dẫn đầu sau từng phiếu: `2, 2, 2, ...`; sau phiếu cuối (`t = 6`) hòa `2-2`, phiếu gần nhất cho `1`.
- Mức 2: `wins = [2, 2, 2, 1]`.
- Mức 3: `[2, 2, 2, 2, 2, 1, 1]`.

**Bài 3**
- Mức 1: `if cnt[cur] <= cnt[p]: cur = p`.
- Mức 2: `i = bisect_right(self.times, t) - 1`.
- Mức 3: `return self.wins[i]`.

**Bài 4**
- Mức 1: lưu hai danh sách `change_times`, `change_leader`.
- Mức 2: thêm mục khi người dẫn đầu đổi (và ở phiếu đầu).
- Mức 3: `q`: `change_leader[bisect_right(change_times, t) - 1]`.

**Bài 5**
- Mức 1: tính `wins[i]` người dẫn đầu từ `times[i]` tới `times[i + 1] - 1`.
- Mức 2: độ dài đoạn `i` là `times[i + 1] - times[i]` (đoạn cuối là `end - times[-1]`).
- Mức 3: cộng độ dài vào `result[wins[i]]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [29]:
print(bisect_right([0, 5, 10], 5), bisect_left([0, 5, 10], 5))
print(bisect_right([0, 5, 10], 7) - 1)
cnt = Counter([0, 1, 1]); print(cnt[0], cnt[1], cnt[2])

assert (bisect_right([0, 5, 10], 5), bisect_left([0, 5, 10], 5)) == (2, 1)
assert bisect_right([0, 5, 10], 7) - 1 == 1
assert (cnt[0], cnt[1], cnt[2]) == (1, 2, 0)

2 1
1
1 2 0


### Đáp án Bài 2

Sau các phiếu `[2, 2, 1, 1]`: `wins = [2, 2, 2, 1]` (sau phiếu cuối hòa `2-2` và phiếu gần nhất là của `1`).

In [30]:
persons, times = [2, 2, 1, 1], [1, 3, 4, 6]
hand_answers_solution = [2, 2, 2, 2, 2, 1, 1]
assert [cls_final(persons, times).q(t) for t in range(1, 8)] == hand_answers_solution
assert oracle_answer(persons, times, list(range(1, 8))) == hand_answers_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [31]:
class ExerciseTopVoted3Solution:

    def __init__(self, persons: List[int], times: List[int]):
        self.times = times
        self.wins = []
        cnt = Counter()
        cur = 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        return self.wins[bisect_right(self.times, t) - 1]

obj = ExerciseTopVoted3Solution(*example_args[0])
assert [obj.q(a[0]) for a in example_args[1:]] == example_expected[1:]
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 10)
    persons = [rng.randrange(n) % 3 for _ in range(n)]
    times = sorted(rng.sample(range(0, 40), n))
    qs = list(range(times[0], 45))
    obj = ExerciseTopVoted3Solution(persons, times)
    assert [obj.q(t) for t in qs] == oracle_answer(persons, times, qs)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [32]:
class ExerciseTopVoted4Solution:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.change_times, self.change_leader = [], []
        cur = None
        for p, tm in zip(persons, times):
            cnt[p] += 1
            if cur is None or cnt[p] >= cnt[cur]:
                if p != cur:
                    self.change_times.append(tm)
                    self.change_leader.append(p)
                cur = p

    def q(self, t: int) -> int:
        return self.change_leader[bisect_right(self.change_times, t) - 1]

obj = ExerciseTopVoted4Solution(*example_args[0])
assert [obj.q(a[0]) for a in example_args[1:]] == example_expected[1:]
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 10)
    persons = [rng.randrange(n) % 3 for _ in range(n)]
    times = sorted(rng.sample(range(0, 40), n))
    qs = list(range(times[0], 45))
    obj = ExerciseTopVoted4Solution(persons, times)
    assert [obj.q(t) for t in qs] == oracle_answer(persons, times, qs)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [33]:
def exercise_5_durations_solution(persons: List[int], times: List[int], end: int) -> dict:
    cnt = Counter()
    cur = 0
    wins = []
    for p in persons:
        cnt[p] += 1
        if cnt[cur] <= cnt[p]:
            cur = p
        wins.append(cur)
    res = {}
    for i, w in enumerate(wins):
        nxt = times[i + 1] if i + 1 < len(times) else end
        length = max(0, min(nxt, end) - times[i])
        if length > 0:
            res[w] = res.get(w, 0) + length
    return res

def durations_oracle(persons, times, end):
    ts = list(range(times[0], end))
    leaders = oracle_answer(persons, times, ts)
    res = {}
    for w in leaders:
        res[w] = res.get(w, 0) + 1
    return res

assert exercise_5_durations_solution([2, 2, 1, 1], [1, 3, 4, 6], 8) == {2: 5, 1: 2} == durations_oracle([2, 2, 1, 1], [1, 3, 4, 6], 8)
print("ví dụ đề (end = 35):", exercise_5_durations_solution(*example_args[0], 35), durations_oracle(*example_args[0], 35))
assert exercise_5_durations_solution(*example_args[0], 35) == {0: 20, 1: 15}
assert exercise_5_durations_solution(*example_args[0], 35) == durations_oracle(*example_args[0], 35)
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 10)
    persons = [rng.randrange(n) % 3 for _ in range(n)]
    times = sorted(rng.sample(range(0, 40), n))
    end = times[-1] + rng.randint(1, 10)
    assert exercise_5_durations_solution(persons, times, end) == durations_oracle(persons, times, end)
print("Bài 5 OK")

ví dụ đề (end = 35): {0: 20, 1: 15} {0: 20, 1: 15}
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tiền xử lý người dẫn đầu + `bisect_right`)

```python
class TopVotedCandidate:

    def __init__(self, persons: List[int], times: List[int]):
        cnt = Counter()
        self.times = times
        self.wins = []                       # wins[i] = người dẫn đầu sau lá phiếu i
        cur = 0
        for p in persons:
            cnt[p] += 1
            if cnt[cur] <= cnt[p]:           # >= : hòa thì phiếu gần nhất (người vừa bỏ phiếu) thắng
                cur = p
            self.wins.append(cur)

    def q(self, t: int) -> int:
        i = bisect_right(self.times, t) - 1  # lá phiếu cuối cùng có thời điểm <= t (tính cả đúng t)
        return self.wins[i]
```

Công thức nhớ nhanh: `wins[i]` tính một lượt; `q(t) = wins[bisect_right(times, t) - 1]`.

### Checklist trước khi submit

- [ ] Hòa phiếu: dùng `>=` (người vừa nhận phiếu thắng)?
- [ ] `bisect_right` (không `bisect_left`) và trừ `1`?
- [ ] Tiền xử lý trong `__init__` (không đếm lại ở mỗi `q`)?
- [ ] Kiểm thử bằng bộ chạy chuỗi lệnh `run_ops` với oracle đếm lại?
- [ ] Đã thử: một phiếu, truy vấn đúng tại/ngay trước/ngay sau thời điểm phiếu, hòa nhiều lần?
- [ ] Tên lớp đúng `TopVotedCandidate`, method `q(self, t)` trả `int`?